In [1]:
import utils
import importlib
importlib.reload(utils) # Ensures the new changes are loaded

C:\Anaconda3\envs\lang_chain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\HOPE\AI CLASS\CHAT GPT API-ADVANCED TECHNIQUES (GENERATIVE AI)\GEMINI\MY HANDS ON\utils.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


<module 'utils' from 'E:\\HOPE\\AI CLASS\\CHAT GPT API-ADVANCED TECHNIQUES (GENERATIVE AI)\\GEMINI\\MY HANDS ON\\utils.py'>

In [2]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

API_KEY = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=API_KEY)

MODEL_NAME = "gemini-3.1-flash-lite-preview"

model = genai.GenerativeModel(MODEL_NAME)

In [3]:
def get_completion(prompt, temperature=0):

    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": temperature
        }
    )

    return response.text

In [4]:
def eval_with_llm(test_set, assistant_answer, debug=False):

    cust_msg = test_set['customer_msg']
    context = test_set['context']
    completion = assistant_answer

    prompt = f"""
You are an expert evaluator.

Your task is to evaluate whether the assistant answer is correct.

Customer message:
{cust_msg}

Context:
{context}

Assistant answer:
{completion}

Respond with only one word:
CORRECT
or
INCORRECT
"""

    response = get_completion(prompt)

    if debug:
        print("Evaluation Response:", response)

    if "CORRECT" in response.upper():
        return 1
    else:
        return 0

In [5]:
test_set = [
    {
        "customer_msg": "What smartphones do you have?",
        "context": """
Available smartphones:
SmartX ProPhone
SmartX MiniPhone
""",
    },
    {
        "customer_msg": "Do you sell gaming consoles?",
        "context": """
Gaming Consoles:
GameSphere X
GameSphere Y
""",
    }
]

In [6]:
assistant_answers = [
    "We offer SmartX ProPhone and SmartX MiniPhone.",
    "Yes, we sell GameSphere X and GameSphere Y."
]

In [7]:
score = 0

for i in range(len(test_set)):

    print("Example", i)

    result = eval_with_llm(
        test_set[i],
        assistant_answers[i],
        debug=True
    )

    print("Score:", result)

    score += result

print("Final Accuracy:", score / len(test_set))

Example 0
Evaluation Response: CORRECT
Score: 1
Example 1
Evaluation Response: CORRECT
Score: 1
Final Accuracy: 1.0


In [8]:
def get_completion_from_messages(messages, temperature=0):

    prompt = ""

    for m in messages:
        role = m["role"]
        content = m["content"]
        prompt += f"{role.upper()}:\n{content}\n\n"

    response = model.generate_content(
        prompt,
        generation_config={"temperature": temperature}
    )

    return response.text.strip()

In [9]:
def eval_vs_ideal(test_set, assistant_answer):

    cust_msg = test_set['customer_msg']
    ideal = test_set['ideal_answer']
    completion = assistant_answer
    
    system_message = """
You are an assistant that evaluates how well the customer service agent 
answers a user question by comparing the response to the ideal (expert) response.
Output a single letter and nothing else.
"""

    user_message = f"""
You are comparing a submitted answer to an expert answer on a given question. Here is the data:

[BEGIN DATA]
************
[Question]: {cust_msg}
************
[Expert]: {ideal}
************
[Submission]: {completion}
************
[END DATA]

Compare the factual content of the submitted answer with the expert answer.
Ignore any differences in style, grammar, or punctuation.

The submitted answer may either be a subset or superset of the expert answer,
or it may conflict with it.

Determine which case applies.

Answer by selecting one option:

(A) The submitted answer is a subset of the expert answer and fully consistent.
(B) The submitted answer is a superset of the expert answer and fully consistent.
(C) The submitted answer contains exactly the same details.
(D) There is disagreement between the answers.
(E) The answers differ but not in a factually important way.

Output only one letter: A, B, C, D, or E.
"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    response = get_completion_from_messages(messages)

    return response

In [10]:
test_set = {
    "customer_msg": "What smartphones do you have?",
    "ideal_answer": "We sell SmartX ProPhone and SmartX MiniPhone."
}

In [12]:
assistant_answer = "Our smartphones include SmartX ProPhone and SmartX MiniPhone."

In [14]:
result = eval_vs_ideal(test_set, assistant_answer)

print(result)

C
